**Import the libraries**

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

print("Libraries imported successfully!")

Libraries imported successfully!


**Loading the dataset**

In [2]:
vocab_size = 10000
max_length = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


**Preprocessing the dataset**

In [3]:
x_train = pad_sequences(
    x_train, maxlen=max_length
)

x_test = pad_sequences(
    x_test, maxlen=max_length
)

print("Data preprocessing completed!")
print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

Data preprocessing completed!
Training shape: (25000, 200)
Testing shape: (25000, 200)


**Creating CNN-LSTM model**

In [6]:
model = Sequential([
    Embedding(10000, 64),

    Conv1D(64, 5, activation='relu'),

    MaxPooling1D(pool_size=2),

    LSTM(64),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.build(input_shape=(None, 200))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 196, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 98, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 693,633 (2.65 MB)

 Trainable params: 693,633 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

**Training the model**

In [7]:
history = model.fit(
    x_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 48s 145ms/step - accuracy: 0.7843 - loss: 0.4384 - val_accuracy: 0.8660 - val_loss: 0.3154
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 142ms/step - accuracy: 0.9158 - loss: 0.2206 - val_accuracy: 0.8608 - val_loss: 0.3473
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 84s 147ms/step - accuracy: 0.9481 - loss: 0.1442 - val_accuracy: 0.8710 - val_loss: 0.3394


**Evaluating the model**

In [8]:
loss, accuracy = model.evaluate(
    x_test, y_test
)

print("Test Accuracy:", accuracy * 100, "%")
print("Test Loss:", loss)

782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.8615 - loss: 0.3627
Test Accuracy: 86.1519992351532 %
Test Loss: 0.3626601994037628


**Saving the trained model**

In [9]:
model.save("movie_sentiment_model.keras")

print("Model saved successfully!")

Model saved successfully!


**Testing own movie reviews**

In [10]:
word_index = imdb.get_word_index()

def predict_review(review):
    words = review.lower().split()
    sequence = [1]

    for word in words:
        index = word_index.get(word, 0)

        if index == 0 or index + 3 >= vocab_size:
            sequence.append(2)
        else:
            sequence.append(index + 3)

    padded = pad_sequences(
        [sequence], maxlen=max_length
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    if prediction >= 0.5:
        print("Positive Review 😊")
    else:
        print("Negative Review 😞")


predict_review("This movie was amazing and entertaining")
predict_review("This movie was boring and disappointing")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Positive Review 😊
Negative Review 😞
